# RAG Anything — Create Sample Input Dataset

This notebook uses **OpenAI web search** and/or **Google Custom Search** to discover downloadable sample assets and place a few cases of each supported type into `./data/raw/showcase`. After the download step, it builds a thesaurus-style inventory over the local dataset.


## API Prerequisites

At least one of these providers should be configured in your `.env`:

- OpenAI web search via `OPENAI_API_KEY`
- Google Programmable Search via `GOOGLE_SEARCH_API_KEY` and `GOOGLE_SEARCH_ENGINE_ID`

Relevant official docs used for this workflow:

- OpenAI web search in the Responses API
- Google Custom Search JSON API / Programmable Search Engine


In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.notebook_utils import ensure_project_root_on_path, ensure_dir, RAW_DATA

ensure_project_root_on_path()
ensure_dir(RAW_DATA)

from src.sample_dataset import create_sample_input_dataset, provider_status
from src.create_thesaurus import create_thesaurus
from src.document_inventory import showcase_coverage

SHOWCASE_ROOT = ensure_dir(RAW_DATA / "showcase")
print(f"Showcase root: {SHOWCASE_ROOT}")


## Check Provider Availability


In [ ]:
status = provider_status()
print(json.dumps(status, indent=2))


## Download A Few Samples Per Group

This step attempts to download up to 3 files for each binary-heavy group into `./data/raw/showcase`.

Current target groups:

- pdf
- word
- excel
- powerpoint
- images
- videos
- audio

The helper keeps the download logic simple and only saves URLs whose path already ends with the expected extension.


In [ ]:
download_result = create_sample_input_dataset(
    SHOWCASE_ROOT,
    max_per_group=3,
    providers=[provider for provider, enabled in status.items() if enabled],
)

print(json.dumps({
    "root": download_result["root"],
    "providers": download_result["providers"],
    "download_count": len(download_result["downloads"]),
}, indent=2))


In [ ]:
pd.DataFrame(download_result["downloads"]).head(100) if download_result["downloads"] else pd.DataFrame(columns=["group", "query", "provider", "url", "target_path", "status", "detail"])


## Add Text And Code Cases If Needed

Search APIs are mainly used above for binary and media examples. For text-like cases, it is often faster to create or copy local examples manually under:

- `./data/raw/showcase/markdown`
- `./data/raw/showcase/code`
- `./data/raw/showcase/text_chunks`

The next cells inventory whatever is present locally.


## Create The Thesaurus From Local Data


In [ ]:
thesaurus_summary = create_thesaurus(SHOWCASE_ROOT)
print(json.dumps({
    "total_files": thesaurus_summary["total_files"],
    "total_size_bytes": thesaurus_summary["total_size_bytes"],
    "groups": thesaurus_summary["groups"],
}, indent=2))


In [ ]:
pd.DataFrame(thesaurus_summary["files"]).head(100) if thesaurus_summary["files"] else pd.DataFrame(columns=["path", "name", "extension", "group", "size_bytes"])


In [ ]:
coverage = showcase_coverage([
    type("Entry", (), item)() for item in thesaurus_summary["files"]
])
coverage_rows = []
for label, info in coverage.items():
    coverage_rows.append({
        "label": label,
        "required": info["required"],
        "targets": ", ".join(info["targets"]),
        "available_total": sum(info["available"].values()),
        "available_detail": json.dumps(info["available"]),
        "ready": info["meets_requirement"],
    })

pd.DataFrame(coverage_rows)


In [ ]:
print(json.dumps(thesaurus_summary["thesaurus"], indent=2)[:12000])


## Next Step

After this dataset bootstrap step, open `./notebooks/s01_search_download_ingestion_showcase.ipynb` to run search, optional further download, ingestion, and thesaurus inspection over the local showcase dataset.
